# 04 · Context capacity: when the image can repair the context model

**Failure being addressed** (notebook 03, validation A5). If the image encodes a function `g(C)` of the context that the context model cannot represent, the joint model uses V to repair `q(D|C)`. The held-out contrast then reports information in a world that has none (`KL_C > 0` in `E Δ̂ = I − KL_VC + KL_C`). In notebook 03 the permutation test rejected a true null in 100 % of replicates, and the polynomial adequacy check flagged only 39 %. Realistic images do encode temperature and time of day, so this case is the norm, not an edge case.

**Remedies compared** (identical context given to both models in every case):

| variant | context columns |
|---|---|
| `base` | hour Fourier basis (2 harmonics) + linear T (notebooks 01–03) |
| `spline` | hour basis + cubic B-spline in T on a fixed physical knot grid (−4…44 °C, every 4 K) + T × hour-basis products |
| `proj` | `base` + `ctxproj_*`: cross-fitted gradient-boosting estimate of E[V | hour, T] (labels never used) |
| `spline+proj` | both |
| `spline+proj/light` | both, with the context block penalised 100× less than the visual block (`ctx_pen = 0.01`, fixed a priori); added after the first development run showed shrinkage of large context effects leaking into the visual coefficient |

With the projection in the context, the joint model can gain only from `V − E[V|C]`. The permutation test's linear residualisation on that context also removes the non-linear context structure that broke its exchangeability.

**Diagnostics.** `flex`: a gradient-boosting Gaussian context model on raw (hour, T) against the variant's context model. `proj`: the variant's context model plus `ctxproj_*` against the variant's context model. Each is flagged when its one-sided 95 % lower bound is above 0. For `proj` variants the second diagnostic is moot.

**Protocol.** Development worlds use new seeds (5000+) and include non-linearities of several shapes. Acceptance criteria for a fresh validation suite (new worlds, seeds ≥ 100 000) are declared in §4 *after* the development run and *before* the validation run. Notebook 03's validation worlds are not reused.

In [1]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from visual_hygrometry import inference, policy, scoring, splits, synthetic, worlds
from visual_hygrometry.synthetic import Node
from visual_hygrometry.worlds import HOUR, cyc, frame, gauss_world, regime

pd.set_option('display.precision', 4)
par = lambda: Parallel(n_jobs=16, max_nbytes=None)
KNOTS = np.arange(-4, 45, 4.0)  # °C, fixed before any run
VARIANTS = {'base': (False, False, 1.0), 'spline': (True, False, 1.0), 'proj': (False, True, 1.0), 'spline+proj': (True, True, 1.0), 'spline+proj/light': (True, True, 0.01)}

In [2]:
def one_rep(W, seed, variants=tuple(VARIANTS), n_pop=200, n_perm=39):
    n = W['n_days']
    df = frame(W, n + n_pop, seed)
    g, slot = df.day.to_numpy(), df.groupby('day').cumcount().to_numpy()
    s = g < n
    raw = df[['hour_local', 'temp_cond_c']].to_numpy()
    df, spl = worlds.add_context_basis(df, KNOTS)
    Vhat = inference.project(raw, df[W['vis']].to_numpy(), g)
    for j, v in enumerate(W['vis']): df[f'ctxproj_{v}'] = Vhat[:, j]
    y = df.d_ref_k.to_numpy()
    sp = splits.kfold(splits.assign_folds(g[s], 5))
    ls_flex = scoring.crossfit(raw[s], y[s], sp, g[s], model=inference.boost)[2]
    half = splits.assign_folds(g[s], 2)
    fit = np.zeros(len(g), bool)
    fit[np.flatnonzero(s)[half == 0]] = True
    out = {'seed': seed}
    for name in variants:
        use_spline, use_proj, cpen = VARIANTS[name]
        ctx = (spl if use_spline else HOUR + ['temp_cond_c']) + ([f'ctxproj_{v}' for v in W['vis']] if use_proj else [])
        Xc, Xvc, _, _ = policy.select_sci(df, 'e2', ctx, W['vis'], allow_shared_sensor=W.get('shared', False))
        C, X = Xc.to_numpy(), Xvc.to_numpy()
        kw = dict(ctx_cols=C.shape[1], ctx_pen=cpen)
        r = scoring.contrast(C[s], X[s], y[s], sp, g[s], **kw)
        _, p, _ = inference.perm_test(C[s], X[s, C.shape[1]:], y[s], sp, g[s], slot[s], n_perm, seed + 500_000, **kw)
        fz = scoring.contrast(C, X, y, [(fit, ~fit)], g, **kw)
        dz = np.full(len(g), np.nan)
        dz[fz.scored] = fz.d
        ev = s & ~fit
        lo, hi = inference.cluster_interval(dz[ev], g[ev])
        adq = inference.adequacy(C[s], y[s], sp, g[s], raw[s], None if use_proj else Vhat[s], ls_flex=ls_flex)
        out |= {f'{name}:est': r.mean, f'{name}:p': p, f'{name}:frz_lo1': inference.cluster_lower(dz[ev], g[ev]), f'{name}:frz_lo': lo, f'{name}:frz_hi': hi,
                f'{name}:frz_target': np.nanmean(dz[~s]), f'{name}:flex': adq['flex'] > 0, f'{name}:projflag': adq.get('proj', -1) > 0}
    return out


def run(W, seeds, variants=tuple(VARIANTS)): return pd.DataFrame(par()(delayed(one_rep)(W, s, variants) for s in seeds))


def summarise(name, W, a, variants=tuple(VARIANTS)):
    t, rows = W['truth'], []
    for v in variants:
        rows.append(dict(world=name, variant=v, truth=t, bias=a[f'{v}:est'].mean() - t, rej_perm=(a[f'{v}:p'] <= .05).mean(), rej_frz=(a[f'{v}:frz_lo1'] > 0).mean(),
                         cov_frz=((a[f'{v}:frz_lo'] <= t) & (a[f'{v}:frz_hi'] >= t)).mean(),
                         cov_frz_target=((a[f'{v}:frz_lo'] <= a[f'{v}:frz_target']) & (a[f'{v}:frz_hi'] >= a[f'{v}:frz_target'])).mean(),
                         flag_flex=a[f'{v}:flex'].mean(), flag_proj=a[f'{v}:projflag'].mean() if 'proj' not in v else np.nan,
                         flag_any=(a[f'{v}:flex'] | a[f'{v}:projflag']).mean()))
    return rows

## 1 · Development worlds

60 days, 48 daylight samples per day, 100 replicates each. `truth` is exact. N3b–N7 have true information 0 but a context non-linearity that the cue encodes.

| world | structure | truth |
|---|---|---|
| N1 | well-specified null | 0 |
| N2 | strong serial dependence null | 0 |
| N4 | multimodal (regime) null | 0 |
| N3b | `D += 0.15 (T−20)²`, cue `(T−20)²/10 + noise` | 0 |
| N6 | `D += 2 σ((T−22)/0.7)` (logistic step), cue = step + noise | 0 |
| N7 | `D += 0.3 (T−20) cos(4π(h−9)/24)` (second-harmonic hour × T), cue = that term/3 + noise | 0 |
| P1 | Gaussian positive | 0.05 |
| P3 | mixed cue: `D += 0.15 (T−20)²`, cue `(T−20)²/10 + 0.081 r + noise(0.5)` | 0.05 |

In [3]:
def nonlin(dterm, vterm, v_sd, gamma=0.0):
    def f(df, rng):
        T, h = df.temp_cond_c.to_numpy(), df.hour.to_numpy()
        df['d_ref_k'] = df.d_ref_k + dterm(T, h)
        df['vis_ctx'] = vterm(T, h) + gamma * df.dres + rng.normal(0, v_sd, len(df))
        return df
    return f

quad = lambda T, h: 0.15 * (T - 20) ** 2
step = lambda T, h: 2 / (1 + np.exp(-(T - 22) / 0.7))
inter = lambda T, h: 0.3 * (T - 20) * np.cos(4 * np.pi * (h - 9) / 24)
g_p3 = np.sqrt(np.expm1(0.1) * 0.25 / 4)
DEV = {'N1': dict(spec=gauss_world(0), vis=['vis_moist'], truth=0.0),
       'N2': dict(spec=gauss_world(0, r_day=.85, r_ar=.97, v_day=.6, v_ar=.9), vis=['vis_moist'], truth=0.0),
       'N4': dict(spec=gauss_world(0), vis=['vis_moist'], truth=0.0, modify=regime((2.5, 0.7))),
       'N3b': dict(spec=gauss_world(0), vis=['vis_ctx'], truth=0.0, modify=nonlin(quad, lambda T, h: (T - 20) ** 2 / 10, .5)),
       'N6': dict(spec=gauss_world(0), vis=['vis_ctx'], truth=0.0, modify=nonlin(step, step, .3)),
       'N7': dict(spec=gauss_world(0), vis=['vis_ctx'], truth=0.0, modify=nonlin(inter, lambda T, h: inter(T, h) / 3, .3)),
       'P1': dict(spec=gauss_world(.162), vis=['vis_moist'], truth=synthetic.cmi(gauss_world(.162), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'P3': dict(spec=gauss_world(0), vis=['vis_ctx'], truth=0.5 * np.log1p(g_p3 ** 2 * 4 / 0.25), modify=nonlin(quad, lambda T, h: (T - 20) ** 2 / 10, .5, g_p3))}
DEV = {k: W | {'n_days': 60} for k, W in DEV.items()}
dev_raw = {k: run(W, range(5000, 5100)) for k, W in DEV.items()}
dev = pd.DataFrame([r for k, a in dev_raw.items() for r in summarise(k, DEV[k], a)]).set_index(['world', 'variant'])
dev

truth        bias  rej_perm  rej_frz  cov_frz  \
world variant                                                             
N1    base               0.0000 -2.3228e-03      0.03     0.02     0.90   
      spline             0.0000 -1.9326e-03      0.03     0.03     0.87   
      proj               0.0000 -2.4214e-03      0.03     0.03     0.89   
      spline+proj        0.0000 -1.7355e-03      0.02     0.03     0.87   
      spline+proj/light  0.0000 -3.9984e-04      0.02     0.01     0.96   
N2    base               0.0000 -6.1669e-03      0.03     0.00     0.87   
      spline             0.0000 -2.4564e-03      0.02     0.06     0.88   
      proj               0.0000 -6.3428e-03      0.03     0.01     0.87   
      spline+proj        0.0000 -3.1301e-03      0.02     0.07     0.85   
      spline+proj/light  0.0000 -1.1356e-02      0.04     0.06     0.92   
N4    base               0.0000 -2.3930e-03      0.01     0.00     0.87   
      spline             0.0000 -1.5177e-03      0.01     0.01     0.89   
      proj               0.0000 -2.3405e-03      0.01     0.00     0.88   
      spline+proj        0.0000 -1.6039e-03      0.01     0.01     0.86   
      spline+proj/light  0.0000 -6.0537e-04      0.06     0.03     0.88   
N3b   base               0.0000  5.9915e-01      1.00     1.00     0.00   
      spline             0.0000  9.5059e-04      0.04     0.08     0.88   
      proj               0.0000  8.7956e-03      0.36     0.08     0.90   
      spline+proj        0.0000  4.0629e-04      0.04     0.07     0.86   
      spline+proj/light  0.0000 -1.2064e-03      0.05     0.18     0.86   
N6    base               0.0000  1.0138e-02      0.86     0.35     0.74   
      spline             0.0000 -8.6837e-04      0.07     0.04     0.84   
      proj               0.0000 -3.3694e-04      0.04     0.01     0.82   
      spline+proj        0.0000 -1.0460e-03      0.06     0.06     0.88   
      spline+proj/light  0.0000  1.8708e-05      0.04     0.12     0.91   
N7    base               0.0000  1.4909e-02      0.99     0.58     0.54   
      spline             0.0000 -2.7798e-04      0.05     0.04     0.90   
      proj               0.0000 -1.2834e-04      0.07     0.05     0.95   
      spline+proj        0.0000 -1.5313e-04      0.05     0.02     0.91   
      spline+proj/light  0.0000 -1.3738e-05      0.04     0.02     0.94   
P1    base               0.0499 -1.7748e-03      1.00     0.70     0.93   
      spline             0.0499 -2.8066e-03      0.98     0.71     0.91   
      proj               0.0499 -1.7907e-03      1.00     0.70     0.94   
      spline+proj        0.0499 -3.0516e-03      0.98     0.72     0.91   
      spline+proj/light  0.0499 -2.3107e-02      0.85     0.61     0.56   
P3    base               0.0500  6.5979e-01      1.00     1.00     0.00   
      spline             0.0500  2.0745e-02      0.99     0.91     0.89   
      proj               0.0500  2.9257e-02      1.00     0.98     0.88   
      spline+proj        0.0500  1.4026e-02      0.98     0.93     0.86   
      spline+proj/light  0.0500 -1.2702e+00      0.89     0.74     0.66   

                         cov_frz_target  flag_flex  flag_proj  flag_any  
world variant                                                            
N1    base                         0.98       0.00       0.02      0.02  
      spline                       0.96       0.00       0.02      0.02  
      proj                         0.97       0.00        NaN      0.00  
      spline+proj                  0.96       0.00        NaN      0.00  
      spline+proj/light            0.96       0.00        NaN      0.00  
N2    base                         0.91       0.00       0.01      0.01  
      spline                       0.88       0.00       0.02      0.02  
      proj                         0.91       0.00        NaN      0.00  
      spline+proj                  0.89       0.00        NaN      0.00  
      spline+proj/light            0.89       0.00        NaN

## 2 · Reading the development suite

*Written after the development run, before the validation run.*

- **`base`** fails exactly as in notebook 03 whenever the cue encodes a context non-linearity. Permutation rejects 1.00 / 0.86 / 0.99 in N3b / N6 / N7, all with true information 0.
- **`spline`** alone fixes the quadratic, the step and the hour × T interaction (permutation 0.04 / 0.07 / 0.05), but the mixed cue P3 is biased +0.021.
- **`proj`** alone fixes the step and the interaction (0.04 / 0.07). It leaves the quadratic at 0.36: the boosted projection approximates g(T) imperfectly, and V exploits the smooth leftover. P3 is biased +0.029.
- **`spline+proj`**: permutation Type I 0.02–0.06 in all six nulls, frozen 0.01–0.07; |bias| ≤ 0.003 in nulls and P1; power 0.98 at P1. The remaining failure is the mixed cue P3, biased **+0.014** (a development check outside the notebook with 150 other seeds gave +0.024 ± 0.006, and +0.008 ± 0.010 without ridge shrinkage). Part of a large context effect is still shrunk into the visual coefficient.
- **`spline+proj/light`** (context block penalised 100× less) was tried to remove that shrinkage and is **rejected**: P3 −1.27 from catastrophic rows, P1 −0.023 with power 0.85.
- **Diagnostics.** On the `base` context, the projection diagnostic flags N3b / N6 / N7 in 100 / 43 / 96 % of replicates; the boosted context model (`flex`) flags 99 / 0 / 0 %. The projection diagnostic is the sensitive one. Neither fires on adequate-context nulls or P1 (≤ 2 %). On `spline+proj` neither fires anywhere.
- **Frozen hold-out** coverage of its own target falls to 0.76 / 0.79 in N3b / P3 under the spline variants (0.89–0.98 elsewhere). Unexplained; flagged.
- **Numerical failure in the first development run:** edge spline columns with almost no training support exploded on held-out rows (biases of ±10³ nats). Fixed by an SD floor in standardisation (`scoring.SD_FLOOR = 0.05`, unit-tested); no earlier world had such columns.

## 3 · Fresh validation suite

**Chosen method:** `spline+proj`, with the permutation test (Q0), the frozen hold-out interval (Q1) and both adequacy diagnostics run on the `base` context as a check that the diagnostics detect inadequacy. `base` is evaluated alongside for reference.

**Criteria, declared before the first validation run** (150 replicates per world; MC SE of a 0.05 rate ≈ 0.018):

| id | criterion (for `spline+proj` unless stated) |
|---|---|
| B1 | permutation Type I ≤ 0.085 in each null X-N1 … X-N6 |
| B2 | frozen Type I ≤ 0.085 in each null |
| B3 | permutation power ≥ 0.80 in X-P1 and X-P3 |
| B4 | |bias| ≤ 0.01 in X-P1, X-P2 (plain cue) and ≤ 0.015 in X-P3, X-P4 (mixed cue) |
| B5 | frozen interval covers the truth ≥ 0.88 and its own target ≥ 0.88 in every positive world |
| B6 | on the `base` context, a diagnostic (`flex` or `proj`) flags ≥ 0.90 of replicates in each of X-N3 … X-N6 |
| B7 | diagnostics on `spline+proj` flag ≤ 0.10 in X-N1, X-N2, X-P1 |

Worlds: 40 samples/day over 06–19 h, 80 days; β = 0.5, T anomaly SD 3.5 K, moisture residual SD 1.8 K (65 % day-level, AR 0.9), visual noise 40 % day-level (AR 0.6); seeds 100 000+. The non-linearities differ from development:

| world | structure | truth |
|---|---|---|
| X-N1 | well-specified null | 0 |
| X-N2 | multimodal (regime 2.2 / 0.6 K, p = 0.35) with strong serial dependence | 0 |
| X-N3 | bump `1.5 exp(−(T−17)²/6)`, encoded by the cue | 0 |
| X-N4 | kink `0.4 |T−21|^1.3`, encoded | 0 |
| X-N5 | `1.2 tanh((T−20)/2) sin(2π(h−8)/24)`, encoded | 0 |
| X-N6 | third hour harmonic `cos(6π(h−10)/24)` (outside the 2-harmonic basis), encoded | 0 |
| X-P1, X-P2 | plain cue | 0.05, 0.10 |
| X-P3 | kink in D, cue = kink/2 + moisture signal | 0.05 |
| X-P4 | bump in D, cue = bump + moisture signal | 0.10 |

In [4]:
XP = dict(per_day=40, hours=(6, 19), n_days=80)
xw = lambda gamma=0.0, **kw: gauss_world(gamma, **({'beta': .5, 't_sd': 3.5, 'r_sd': 1.8, 'r_day': .65, 'r_ar': .9, 'v_day': .4, 'v_ar': .6} | kw))
bump = lambda T, h: 1.5 * np.exp(-(T - 17) ** 2 / 6)
kink = lambda T, h: 0.4 * np.abs(T - 21) ** 1.3
tsin = lambda T, h: 1.2 * np.tanh((T - 20) / 2) * np.sin(2 * np.pi * (h - 8) / 24)
harm3 = lambda T, h: np.cos(6 * np.pi * (h - 10) / 24)
cmi_mixed = lambda gamma, v_sd: 0.5 * np.log1p(gamma ** 2 * 1.8 ** 2 / v_sd ** 2)
gam = lambda c, v_sd: np.sqrt(np.expm1(2 * c) * v_sd ** 2) / 1.8
VAL = {'X-N1': dict(spec=xw(), vis=['vis_moist'], truth=0.0),
       'X-N2': dict(spec=xw(r_day=.85, r_ar=.97), vis=['vis_moist'], truth=0.0, modify=regime((2.2, .6), .35)),
       'X-N3': dict(spec=xw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(bump, lambda T, h: bump(T, h) / 1.5 * 2, .4)),
       'X-N4': dict(spec=xw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(kink, lambda T, h: kink(T, h) / 2, .4)),
       'X-N5': dict(spec=xw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(tsin, tsin, .3)),
       'X-N6': dict(spec=xw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(harm3, harm3, .3)),
       'X-P1': dict(spec=xw(gam(.05, 1.0)), vis=['vis_moist'], truth=synthetic.cmi(xw(gam(.05, 1.0)), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'X-P2': dict(spec=xw(gam(.10, 1.0)), vis=['vis_moist'], truth=synthetic.cmi(xw(gam(.10, 1.0)), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'X-P3': dict(spec=xw(), vis=['vis_ctx'], truth=cmi_mixed(gam(.05, .4), .4), modify=nonlin(kink, lambda T, h: kink(T, h) / 2, .4, gam(.05, .4))),
       'X-P4': dict(spec=xw(), vis=['vis_ctx'], truth=cmi_mixed(gam(.10, .4), .4), modify=nonlin(bump, lambda T, h: bump(T, h) / 1.5 * 2, .4, gam(.10, .4)))}
VAL = {k: XP | W for k, W in VAL.items()}
CHOSEN = ['base', 'spline+proj']
val_raw = {k: run(W, range(100000, 100150), CHOSEN) for k, W in VAL.items()}
val = pd.DataFrame([r for k, a in val_raw.items() for r in summarise(k, VAL[k], a, CHOSEN)]).set_index(['world', 'variant'])
val

truth        bias  rej_perm  rej_frz  cov_frz  \
world variant                                                      
X-N1  base          0.00 -2.5784e-03    0.0733   0.0400   0.9000   
      spline+proj   0.00 -1.7906e-03    0.0600   0.0467   0.8933   
X-N2  base          0.00 -3.4722e-03    0.0400   0.0333   0.8800   
      spline+proj   0.00 -2.7179e-03    0.0467   0.0533   0.8867   
X-N3  base          0.00  1.0239e-02    0.9600   0.5000   0.6067   
      spline+proj   0.00 -1.6603e-05    0.0333   0.0867   0.8867   
X-N4  base          0.00  3.2488e-01    1.0000   0.9867   0.0333   
      spline+proj   0.00  2.2430e-02    0.0600   0.0467   0.9267   
X-N5  base          0.00  9.8180e-03    0.8000   0.3000   0.7933   
      spline+proj   0.00 -5.8462e-04    0.0600   0.0600   0.8533   
X-N6  base          0.00  3.2332e-03    0.0467   0.4867   0.6467   
      spline+proj   0.00 -4.6456e-04    0.0600   0.0733   0.9200   
X-P1  base          0.05 -3.8040e-03    0.9733   0.5200   0.9000   
      spline+proj   0.05 -4.8647e-03    0.9733   0.6000   0.8867   
X-P2  base          0.10 -4.1008e-03    1.0000   0.8000   0.9067   
      spline+proj   0.10 -2.6844e-02    0.9800   0.8333   0.9133   
X-P3  base          0.05  3.9261e-01    1.0000   1.0000   0.0133   
      spline+proj   0.05 -1.0536e-01    0.9933   0.9400   0.8933   
X-P4  base          0.10 -3.9311e-05    1.0000   0.9933   0.9467   
      spline+proj   0.10 -2.2246e-01    0.9800   0.9600   0.8667   

                   cov_frz_target  flag_flex  flag_proj  flag_any  
world variant                                                      
X-N1  base                 0.9000        0.0     0.0000    0.0000  
      spline+proj          0.9000        0.0        NaN    0.0000  
X-N2  base                 0.9000        0.0     0.0067    0.0067  
      spline+proj          0.8600        0.0        NaN    0.0000  
X-N3  base                 0.9333        0.0     0.7533    0.7533  
      spline+proj          0.8667        0.0        NaN    0.0000  
X-N4  base                 0.9267        1.0     1.0000    1.0000  
      spline+proj          0.7933        0.0        NaN    0.0000  
X-N5  base                 0.9200        0.0     0.3733    0.3733  
      spline+proj          0.9000        0.0        NaN    0.0000  
X-N6  base                 0.9400        0.0     0.9200    0.9200  
      spline+proj          0.9000        0.0        NaN    0.0000  
X-P1  base                 0.9133        0.0     0.0133    0.0133  
      spline+proj          0.8800        0.0        NaN    0.0000  
X-P2  base                 0.9067        0.0     0.0200    0.0200  
      spline+proj          0.8867        0.0        NaN    0.0000  
X-P3  base                 0.9267        1.0     1.0000    1.0000  
      spline+proj          0.8333        0.0        NaN    0.0000  
X-P4  base                 0.9267        0.0     0.7067    0.7067  
      spline+proj          0.8600        0.0        NaN    0.0000

In [5]:
v = lambda w, c: val.loc[(w, 'spline+proj'), c]
b = lambda w, c: val.loc[(w, 'base'), c]
nulls, plain, mixed = [f'X-N{i}' for i in range(1, 7)], ['X-P1', 'X-P2'], ['X-P3', 'X-P4']
acc = [('B1', 'perm Type I <= 0.085, X-N1..N6', [v(w, 'rej_perm') for w in nulls], all(v(w, 'rej_perm') <= .085 for w in nulls)),
       ('B2', 'frozen Type I <= 0.085, X-N1..N6', [v(w, 'rej_frz') for w in nulls], all(v(w, 'rej_frz') <= .085 for w in nulls)),
       ('B3', 'perm power >= 0.80, X-P1 / X-P3', [v(w, 'rej_perm') for w in ('X-P1', 'X-P3')], all(v(w, 'rej_perm') >= .8 for w in ('X-P1', 'X-P3'))),
       ('B4', '|bias| <= 0.01 plain, <= 0.015 mixed', [v(w, 'bias') for w in plain + mixed], all(abs(v(w, 'bias')) <= .01 for w in plain) and all(abs(v(w, 'bias')) <= .015 for w in mixed)),
       ('B5', 'frozen covers truth and target >= 0.88', [(v(w, 'cov_frz'), v(w, 'cov_frz_target')) for w in plain + mixed], all(v(w, 'cov_frz') >= .88 and v(w, 'cov_frz_target') >= .88 for w in plain + mixed)),
       ('B6', 'base-context diagnostics flag >= 0.90, X-N3..N6', [b(w, 'flag_any') for w in nulls[2:]], all(b(w, 'flag_any') >= .9 for w in nulls[2:])),
       ('B7', 'spline+proj diagnostics flag <= 0.10, X-N1/N2/P1', [v(w, 'flag_any') for w in ('X-N1', 'X-N2', 'X-P1')], all(v(w, 'flag_any') <= .1 for w in ('X-N1', 'X-N2', 'X-P1')))]
pd.DataFrame([(i, c, np.round(x, 3), p) for i, c, x, p in acc], columns=['id', 'criterion', 'value', 'pass'])

,id,criterion,value,pass
0,B1,"perm Type I <= 0.085, X-N1..N6","[0.06, 0.047, 0.033, 0.06, 0.06, 0.06]",True
1,B2,"frozen Type I <= 0.085, X-N1..N6","[0.047, 0.053, 0.087, 0.047, 0.06, 0.073]",False
2,B3,"perm power >= 0.80, X-P1 / X-P3","[0.973, 0.993]",True
3,B4,"|bias| <= 0.01 plain, <= 0.015 mixed","[-0.005, -0.027, -0.105, -0.222]",False
4,B5,frozen covers truth and target >= 0.88,"[[0.887, 0.88], [0.913, 0.887], [0.893, 0.833]...",False
5,B6,"base-context diagnostics flag >= 0.90, X-N3..N6","[0.753, 1.0, 0.373, 0.92]",False
6,B7,"spline+proj diagnostics flag <= 0.10, X-N1/N2/P1","[0.0, 0.0, 0.0]",True


**Validation outcome: B1, B3, B7 pass; B2, B4, B5, B6 fail.** The revised method is not yet acceptable for confirmatory use.

- **Fixed (B1):** the zero-information test is now valid where notebook 03's failed. Permutation Type I is 0.033–0.060 in all six fresh nulls, including the bump, kink, tanh × sin and third-hour-harmonic worlds, where `base` rejected 0.80–1.00. Power is 0.97–0.99 (B3), and the diagnostics never fire on adequate contexts (B7).
- **B2 (narrow):** frozen Type I 0.087 in the bump world, against a criterion of 0.085 (MC SE 0.018).
- **B4 (serious):** the effect-size *mean* is wrong under `spline+proj`: −0.027 at truth 0.10, −0.105 and −0.222 in the mixed-cue worlds. Diagnosed outside the notebook on 60 new replicates (seeds 110 000+, not the validation seeds), it is a tail failure, not a shifted estimator. The median estimate is 0.055 against a truth of 0.05. Some replicates contain single held-out rows scoring −117 to −130 nats at the temperature extremes (2–39 °C, near the knot-grid edges), where the cue is large and the joint model's linear coefficient on V extrapolates. Clipping per-row losses at −5 nats gives a mean of 0.062. The development worlds (narrower T, 60 days) did not reach those extremes.
- **B5:** in the mixed worlds the frozen interval covers its own target in only 0.83 / 0.86 of replicates, consistent with the same heavy-tailed rows.
- **B6:** run on the `base` context, the diagnostics flag the bump world in 0.75 of replicates and the tanh × sin world in only 0.37. They cannot replace a context model with enough capacity; they are a secondary check.

**Next (not done here):** bound per-row losses in the Gaussian family with the same defensive mixture as `ls-kde`, and restrict scoring to rows whose context lies inside the training support (rows outside reported and counted). Develop on the development worlds, then validate on another fresh suite.